<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/04_aggregation_groupby.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 · Aggregation (GROUP BY)

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~40 min**

## Learning objectives
- use `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`;
- summarise per group with `GROUP BY`;
- filter groups with `HAVING`.

Aggregate functions summarise many rows into one number. `GROUP BY` computes one summary per group, which is central to any data analysis.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## Setup — run this cell first

This connects the notebook to the example database. It works both on your own computer and on Google Colab; just run it (Shift+Enter). You do not need to understand it.

In [ ]:
# Run me fiRst. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

## 1. Aggregate the whole table

In [ ]:
%%sql
SELECT COUNT(*) AS n_samples,
       COUNT(DISTINCT environment) AS n_environments
FROM samples;

`MIN`/`MAX`/`AVG` summarise a numeric column over the whole table in one step.

In [ ]:
%%sql
-- the pH range and mean across the whole study (NULLs ignored)
SELECT MIN(ph) AS lowest_ph, MAX(ph) AS highest_ph,
       ROUND(AVG(ph), 2) AS mean_ph
FROM samples;

## 2. One summary per group
How many samples per environment?

In [ ]:
%%sql
SELECT environment, COUNT(*) AS n_samples
FROM samples
GROUP BY environment;

Group by a different column. You can also `GROUP BY` two columns to get one row per *combination*.

In [ ]:
%%sql
-- samples per environment x treatment (a 2-way breakdown)
SELECT environment, treatment, COUNT(*) AS n_samples
FROM samples
GROUP BY environment, treatment
ORDER BY environment, treatment;

## 3. SUM per group — library size
Total reads sequenced in each sample (the abundance table has one row per taxon per sample).

In [ ]:
%%sql
SELECT sample_id, SUM(count) AS total_reads
FROM abundance
GROUP BY sample_id
ORDER BY total_reads DESC
LIMIT 10;

## 4. AVG per group
Mean pH per environment (NULLs are ignored by AVG).

In [ ]:
%%sql
SELECT environment, ROUND(AVG(ph), 2) AS mean_ph
FROM samples
GROUP BY environment;

Several aggregates can sit side by side in one grouped query, giving a compact per-group profile.

In [ ]:
%%sql
-- min, mean and max temperature per environment in one table
SELECT environment,
       ROUND(MIN(temperature_c), 1) AS coldest,
       ROUND(AVG(temperature_c), 1) AS mean_temp,
       ROUND(MAX(temperature_c), 1) AS warmest
FROM samples
GROUP BY environment;

## 5. HAVING — filter the GROUPS
`WHERE` filters rows *before* grouping; `HAVING` filters *after*, using the aggregate.

In [ ]:
%%sql
SELECT sample_id, SUM(count) AS total_reads
FROM abundance
GROUP BY sample_id
HAVING total_reads > 1000
ORDER BY total_reads DESC;

`HAVING` can test a `COUNT` too, for example keep only groups with enough members.

In [ ]:
%%sql
-- phyla represented by at least 4 taxa (filter groups by COUNT)
SELECT phylum, COUNT(*) AS n_taxa
FROM taxa
GROUP BY phylum
HAVING n_taxa >= 4
ORDER BY n_taxa DESC;

---
## Exercises

**Exercise.** Count how many taxa belong to each phylum.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT phylum, COUNT(*) AS n_taxa FROM taxa
GROUP BY phylum
ORDER BY n_taxa DESC;
```
</details>

**Exercise.** Compute the total reads of each taxon (sum over all samples); show the top 10.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT taxon_id, SUM(count) AS total_reads FROM abundance
GROUP BY taxon_id
ORDER BY total_reads DESC
LIMIT 10;
```
</details>

**Exercise.** Which environments have a mean temperature above 15 °C? (GROUP BY + HAVING)

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT environment, ROUND(AVG(temperature_c),1) AS mean_temp FROM samples
GROUP BY environment
HAVING mean_temp > 15;
```
</details>

### Recap
- Aggregates: `COUNT/SUM/AVG/MIN/MAX`; `AVG` ignores NULLs.
- `GROUP BY col` = one summary row per value of `col`.
- `HAVING` filters groups (use it for aggregate conditions).
- Next: 05 · Joining tables.